# Global Trade Macro Story — Verification

**Focus:** The ~$58T peak, the **2009 financial crisis** shock, and the **2015 commodity slowdown**.

This notebook has two parts:
1. **Descriptive analytics** — aggregations, shock stats, charts
2. **Machine learning validation** — 6 models that independently test whether 2009 & 2015 are real structural breaks

Review all outputs, then confirm before building the Streamlit dashboard.

**Dependencies:** `pip install -r requirements.txt`

In [ ]:
import os

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
DATA_PATH = r"D:\Code\GlobalCommodity\CSV Dataset\commodity_trade_statistics_data.csv"
OUTPUT_DIR = r"D:\Code\GlobalCommodity\outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
df = pd.read_csv(
    DATA_PATH,
    low_memory=False,
    dtype={"comm_code": str},
)

# Exclude the catch-all aggregate bucket for the "clean" economic series
df_clean = df[df["category"] != "all_commodities"].copy()

print(f"Rows loaded: {len(df):,}")
print(f"Years: {df['year'].min()} – {df['year'].max()}")
print(f"Rows after excluding 'all_commodities': {len(df_clean):,}")

In [ ]:
def yearly_trade_summary(data: pd.DataFrame, label: str) -> pd.DataFrame:
    summary = (
        data.groupby("year")
        .agg(
            total_usd=("trade_usd", "sum"),
            n_records=("trade_usd", "count"),
            n_countries=("country_or_area", "nunique"),
        )
        .reset_index()
        .sort_values("year")
    )
    summary["total_trillions"] = summary["total_usd"] / 1e12
    summary["yoy_pct"] = summary["total_usd"].pct_change() * 100
    summary["series"] = label
    return summary


yearly_raw = yearly_trade_summary(df, "raw_total")
yearly_clean = yearly_trade_summary(df_clean, "clean_excl_all_commodities")

# Import / export split (clean series)
flow_yearly = (
    df_clean.groupby(["year", "flow"])["trade_usd"]
    .sum()
    .unstack(fill_value=0)
    .reset_index()
)
for col in flow_yearly.columns:
    if col != "year":
        flow_yearly[col] = flow_yearly[col] / 1e12

yearly_clean = yearly_clean.merge(flow_yearly, on="year", how="left")
yearly_clean.head()

In [ ]:
def shock_stats(summary: pd.DataFrame) -> pd.DataFrame:
    peak = summary.loc[summary["total_trillions"].idxmax()]
    rows = []

    def add_event(name, y1, y2):
        v1 = summary.loc[summary["year"] == y1, "total_trillions"].iloc[0]
        v2 = summary.loc[summary["year"] == y2, "total_trillions"].iloc[0]
        rows.append(
            {
                "event": name,
                "from_year": y1,
                "to_year": y2,
                "from_trillions": round(v1, 2),
                "to_trillions": round(v2, 2),
                "change_trillions": round(v2 - v1, 2),
                "change_pct": round((v2 / v1 - 1) * 100, 1),
            }
        )

    rows.append(
        {
            "event": "peak_year",
            "from_year": int(peak["year"]),
            "to_year": None,
            "from_trillions": round(peak["total_trillions"], 2),
            "to_trillions": None,
            "change_trillions": None,
            "change_pct": None,
        }
    )
    add_event("2009_financial_crisis", 2008, 2009)
    add_event("2015_commodity_slowdown", 2014, 2015)
    add_event("post_crisis_recovery", 2009, 2010)
    return pd.DataFrame(rows)


shock_raw = shock_stats(yearly_raw)
shock_clean = shock_stats(yearly_clean)

print("=== HEADLINE (raw total, includes all_commodities) ===")
display(shock_raw)

print("\n=== CLEAN SERIES (excludes all_commodities — more reliable for category analysis) ===")
display(shock_clean)

## Key numbers to verify

| Metric | Raw total | Clean series |
|--------|-----------|--------------|
| **Peak year** | 2013 (~$58.0T) | 2011 (~$15.5T) |
| **2008 → 2009 drop** | −20.7% | −17.0% |
| **2009 → 2010 rebound** | +21.0% | strong recovery |
| **2014 → 2015 drop** | −13.3% | −15.1% |

**Important:** The headline **$58T** figure comes from the raw sum, which includes an `all_commodities` aggregate bucket that double-counts at the global level. The **shape** of the crisis (2009) and slowdown (2015) is real in both series. Mineral fuels (oil) are the largest *real* category driver of both shocks.

**2016 caveat:** Record count drops sharply in 2016 (~290K vs ~360K), so the 2016 dip is partly incomplete reporting — do not treat it as a third macro shock without filtering.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# --- Chart 1: dual series ---
ax = axes[0]
ax.plot(
    yearly_raw["year"],
    yearly_raw["total_trillions"],
    marker="o",
    linewidth=2.5,
    label="Raw total (headline $58T story)",
    color="#1f77b4",
)
ax.plot(
    yearly_clean["year"],
    yearly_clean["total_trillions"],
    marker="s",
    linewidth=2,
    linestyle="--",
    label="Clean total (excl. all_commodities)",
    color="#2ca02c",
)

peak_raw = yearly_raw.loc[yearly_raw["total_trillions"].idxmax()]
ax.annotate(
    f"Peak {int(peak_raw['year'])}\n${peak_raw['total_trillions']:.1f}T",
    xy=(peak_raw["year"], peak_raw["total_trillions"]),
    xytext=(peak_raw["year"] - 4, peak_raw["total_trillions"] + 3),
    arrowprops=dict(arrowstyle="->", color="#333"),
    fontsize=10,
    fontweight="bold",
)

for y1, y2, label, color in [
    (2008, 2009, "2009 crisis\n−20.7%", "#d62728"),
    (2014, 2015, "2015 slowdown\n−13.3%", "#ff7f0e"),
]:
    v1 = yearly_raw.loc[yearly_raw["year"] == y1, "total_trillions"].iloc[0]
    v2 = yearly_raw.loc[yearly_raw["year"] == y2, "total_trillions"].iloc[0]
    mid_y = (v1 + v2) / 2
    ax.annotate(
        label,
        xy=(y2, v2),
        xytext=(y2 + 0.5, mid_y),
        fontsize=9,
        color=color,
        fontweight="bold",
    )

ax.set_ylabel("Total trade (trillions USD)")
ax.set_title("Global Commodity Trade: The $58T Peak and Two Macro Shocks", fontsize=14, fontweight="bold")
ax.legend(loc="upper left")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.0f}T"))

# --- Chart 2: YoY growth ---
ax2 = axes[1]
colors = [
    "#d62728" if y in (2009, 2015, 2016) else "#4c72b0" if y == 2010 else "#7f7f7f"
    for y in yearly_raw["year"]
]
ax2.bar(yearly_raw["year"], yearly_raw["yoy_pct"], color=colors, edgecolor="white", linewidth=0.5)
ax2.axhline(0, color="black", linewidth=0.8)
ax2.set_ylabel("YoY change (%)")
ax2.set_xlabel("Year")
ax2.set_title("Year-over-year growth — red bars = contraction years")

plt.tight_layout()
plt.savefig(r"D:\Code\GlobalCommodity\outputs\01_macro_trade_timeline.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Import vs Export over time (clean series)
fig, ax = plt.subplots(figsize=(14, 5))

ax.fill_between(flow_yearly["year"], 0, flow_yearly.get("Import", 0), alpha=0.6, label="Import", color="#e74c3c")
ax.fill_between(
    flow_yearly["year"],
    flow_yearly.get("Import", 0),
    flow_yearly.get("Import", 0) + flow_yearly.get("Export", 0),
    alpha=0.6,
    label="Export",
    color="#3498db",
)

ax.set_title("Import vs Export Trade Volume (clean series, trillions USD)")
ax.set_ylabel("Trillions USD")
ax.set_xlabel("Year")
ax.legend(loc="upper left")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.0f}T"))

for shock_year in (2009, 2015):
    ax.axvline(shock_year, color="black", linestyle=":", alpha=0.5)

plt.tight_layout()
plt.savefig(r"D:\Code\GlobalCommodity\outputs\02_import_export_stack.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
OIL_CAT = "27_mineral_fuels_oils_distillation_products_etc"


def category_shock(y1: int, y2: int, top_n: int = 10) -> tuple[pd.DataFrame, pd.DataFrame]:
    before = df_clean[df_clean["year"] == y1].groupby("category")["trade_usd"].sum()
    after = df_clean[df_clean["year"] == y2].groupby("category")["trade_usd"].sum()
    delta = (after - before).sort_values()
    out = pd.DataFrame({"change_usd": delta, "change_trillions": delta / 1e12})
    out["change_pct"] = (after / before - 1).reindex(delta.index) * 100
    return out.head(top_n), out.tail(top_n)


decline_09, gain_09 = category_shock(2008, 2009)
decline_15, gain_15 = category_shock(2014, 2015)

print("2009 crisis — biggest category DECLINES (clean series):")
display(decline_09)

print("\n2015 slowdown — biggest category DECLINES (clean series):")
display(decline_15)

# Oil trend overlay
oil_yearly = df_clean[df_clean["category"] == OIL_CAT].groupby("year")["trade_usd"].sum() / 1e12
oil_share = (oil_yearly / yearly_clean.set_index("year")["total_trillions"] * 100).rename("oil_share_pct")

fig, ax1 = plt.subplots(figsize=(14, 5))
ax1.bar(yearly_clean["year"], yearly_clean["total_trillions"], alpha=0.35, color="#95a5a6", label="Total trade (clean)")
ax1.plot(oil_yearly.index, oil_yearly.values, color="#c0392b", marker="o", linewidth=2.5, label="Mineral fuels & oils")
ax1.set_ylabel("Trillions USD")
ax1.set_xlabel("Year")
ax1.set_title("Oil/fuels trade vs total — the engine behind both shocks")
ax1.legend(loc="upper left")

ax2 = ax1.twinx()
ax2.plot(oil_share.index, oil_share.values, color="#8e44ad", linestyle="--", marker="s", label="Oil share of trade")
ax2.set_ylabel("Oil share (%)")
ax2.legend(loc="upper right")

for shock_year in (2009, 2015):
    ax1.axvline(shock_year, color="black", linestyle=":", alpha=0.4)

plt.tight_layout()
plt.savefig(r"D:\Code\GlobalCommodity\outputs\03_oil_drives_shocks.png", dpi=150, bbox_inches="tight")
plt.show()

## Machine learning layer — deeper verification

Six models that **validate** the macro story from different angles. None of them are told that 2009 or 2015 are shock years — we check whether they discover those breaks on their own.

| # | Model | Library | What it answers |
|---|-------|---------|-----------------|
| 1 | **YoY z-score + Isolation Forest** | sklearn | Which years have extreme growth swings? |
| 2 | **K-Means country clustering** | sklearn | Who got hit vs who stayed resilient? |
| 3 | **Change-point detection (PELT)** | ruptures | When did the global growth regime shift? |
| 4 | **ARIMA counterfactual** | statsmodels | How much trade was *lost* vs pre-crisis trend? |
| 5 | **Random Forest YoY surprise** | sklearn | Which years could momentum not explain? |
| 6 | **Oil → total regression** | sklearn | How much of each shock is explained by oil alone? |

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

# --- 1. Trend model on LEVELS (clean series) ---
trend_df = yearly_clean.dropna(subset=["total_trillions"]).copy()
trend_model = LinearRegression()
trend_model.fit(trend_df[["year"]], trend_df["total_trillions"])
trend_df["predicted_trillions"] = trend_model.predict(trend_df[["year"]])
trend_df["residual"] = trend_df["total_trillions"] - trend_df["predicted_trillions"]
trend_df["residual_z"] = (trend_df["residual"] - trend_df["residual"].mean()) / trend_df["residual"].std()
trend_df["level_anomaly"] = trend_df["residual_z"].abs() > 1.5

print("Level trend anomalies (|z| > 1.5) — peaks and incomplete years:")
display(
    trend_df.loc[trend_df["level_anomaly"], ["year", "total_trillions", "predicted_trillions", "residual_z"]]
)

# --- 2. Anomaly detection on YoY GROWTH (better for shocks) ---
yoy_df = yearly_raw.dropna(subset=["yoy_pct"]).copy()
yoy_df["yoy_z"] = (yoy_df["yoy_pct"] - yoy_df["yoy_pct"].mean()) / yoy_df["yoy_pct"].std()
yoy_df["shock_anomaly"] = yoy_df["yoy_z"].abs() > 1.5

print("\nYoY growth anomalies (|z| > 1.5) — crisis & recovery years:")
display(yoy_df.loc[yoy_df["shock_anomaly"], ["year", "total_trillions", "yoy_pct", "yoy_z"]])

# --- 3. Isolation Forest on raw yearly features ---
feature_cols = ["total_trillions", "yoy_pct", "n_records"]
ml_yearly = yearly_raw.dropna(subset=feature_cols).copy()
X_ml = StandardScaler().fit_transform(ml_yearly[feature_cols])
iso = IsolationForest(contamination=0.12, random_state=42)
ml_yearly["iso_anomaly"] = iso.fit_predict(X_ml) == -1

print("\nIsolation Forest (multivariate) flagged years:")
display(ml_yearly.loc[ml_yearly["iso_anomaly"], ["year"] + feature_cols])

# --- Chart: YoY with ML shock flags ---
fig, ax = plt.subplots(figsize=(14, 5))
colors = ["#d62728" if a else "#4c72b0" for a in yoy_df["shock_anomaly"]]
ax.bar(yoy_df["year"], yoy_df["yoy_pct"], color=colors, edgecolor="white")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("ML validation: YoY z-score flags shock & recovery years (red)")
ax.set_ylabel("YoY change (%)")
ax.set_xlabel("Year")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "04_ml_yoy_anomalies.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
def country_shock_pct(y1: int, y2: int) -> pd.Series:
    before = df_clean[df_clean["year"] == y1].groupby("country_or_area")["trade_usd"].sum()
    after = df_clean[df_clean["year"] == y2].groupby("country_or_area")["trade_usd"].sum()
    common = before.index.intersection(after.index)
    return ((after[common] / before[common]) - 1) * 100


shock_09 = country_shock_pct(2008, 2009).rename("shock_2009_pct")
shock_15 = country_shock_pct(2014, 2015).rename("shock_2015_pct")

country_shocks = pd.concat([shock_09, shock_15], axis=1).dropna()
country_shocks = country_shocks.replace([np.inf, -np.inf], np.nan).dropna()
country_shocks = country_shocks[
    (country_shocks["shock_2009_pct"].between(-80, 200))
    & (country_shocks["shock_2015_pct"].between(-80, 200))
]

X_country = StandardScaler().fit_transform(country_shocks)
best_k, best_score = 3, -1
for k in range(2, 6):
    labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X_country)
    score = silhouette_score(X_country, labels)
    if score > best_score:
        best_k, best_score = k, score

kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
country_shocks["cluster"] = kmeans.fit_predict(X_country)

cluster_summary = (
    country_shocks.groupby("cluster")
    .agg(
        n_countries=("shock_2009_pct", "count"),
        avg_shock_2009=("shock_2009_pct", "mean"),
        avg_shock_2015=("shock_2015_pct", "mean"),
    )
    .round(1)
)
print(f"Country shock clusters (k={best_k}, silhouette={best_score:.2f})")
display(cluster_summary)

for c in sorted(country_shocks["cluster"].unique()):
    examples = country_shocks[country_shocks["cluster"] == c].sort_values("shock_2009_pct").head(3).index.tolist()
    print(f"  Cluster {c} examples: {', '.join(examples)}")

# Scatter: 2009 vs 2015 shock by cluster
fig, ax = plt.subplots(figsize=(10, 7))
for c in sorted(country_shocks["cluster"].unique()):
    subset = country_shocks[country_shocks["cluster"] == c]
    ax.scatter(subset["shock_2009_pct"], subset["shock_2015_pct"], label=f"Cluster {c}", alpha=0.7, s=40)

ax.axhline(0, color="black", linewidth=0.5)
ax.axvline(0, color="black", linewidth=0.5)
ax.set_xlabel("2008→2009 trade change (%)")
ax.set_ylabel("2014→2015 trade change (%)")
ax.set_title("Country shock profiles — K-Means clusters")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "05_country_shock_clusters.png"), dpi=150, bbox_inches="tight")
plt.show()

### Model 3 — Change-point detection (ruptures)

Automatically finds years where the **growth regime** shifted, without being told about 2009 or 2015.

In [ ]:
import ruptures as rpt

levels = yearly_raw.sort_values("year")["total_trillions"].values.reshape(-1, 1)
years_arr = yearly_raw.sort_values("year")["year"].values
yoy_vals = yearly_raw.sort_values("year")["yoy_pct"].dropna().values.reshape(-1, 1)
yoy_years = yearly_raw.sort_values("year").dropna(subset=["yoy_pct"])["year"].values

# Regime shifts in trade LEVELS (penalty tuned for 3-5 major breaks)
level_algo = rpt.Pelt(model="l2", min_size=3, jump=1).fit(levels)
level_bkps = level_algo.predict(pen=10)
level_cps = [int(years_arr[i]) for i in level_bkps[:-1]]

# Regime shifts in YoY GROWTH (more sensitive penalty)
yoy_algo = rpt.Pelt(model="rbf", min_size=2, jump=1).fit(yoy_vals)
yoy_bkps = yoy_algo.predict(pen=1)
yoy_cps = [int(yoy_years[i]) for i in yoy_bkps[:-1]]

changepoint_df = pd.DataFrame(
    {
        "series": ["trade_levels", "yoy_growth"],
        "change_years": [level_cps, yoy_cps],
        "n_breaks": [len(level_cps), len(yoy_cps)],
    }
)
print("Detected structural break years:")
display(changepoint_df)

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

ax = axes[0]
ax.plot(years_arr, levels, "o-", color="#1f77b4", linewidth=2)
for cp in level_cps:
    ax.axvline(cp, color="#d62728", linestyle="--", alpha=0.7)
    ax.text(cp + 0.2, levels.max() * 0.95, str(cp), color="#d62728", fontweight="bold")
ax.set_ylabel("Trillions USD")
ax.set_title("Change-points on trade LEVELS (red lines = regime shifts)")

ax2 = axes[1]
ax2.bar(yoy_years, yoy_vals.flatten(), color="#4c72b0", edgecolor="white")
for cp in yoy_cps:
    ax2.axvline(cp, color="#ff7f0e", linestyle="--", linewidth=2)
ax2.axhline(0, color="black", linewidth=0.5)
ax2.set_ylabel("YoY %")
ax2.set_xlabel("Year")
ax2.set_title("Change-points on YoY growth")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "06_ml_changepoints.png"), dpi=150, bbox_inches="tight")
plt.show()

### Model 4 — ARIMA counterfactual

Train on **1988–2007** (pre-crisis expansion), then forecast 2008–2015. The gap between forecast and actual quantifies **trade lost to macro shocks**.

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
import warnings

warnings.filterwarnings("ignore")

series = yearly_raw.sort_values("year").reset_index(drop=True)
train_end_year = 2007
train = series.loc[series["year"] <= train_end_year, "total_trillions"].values
forecast_years = series.loc[series["year"] > train_end_year, "year"].values
actual = series.loc[series["year"] > train_end_year, "total_trillions"].values

arima = ARIMA(train, order=(1, 1, 1))
arima_fit = arima.fit()
forecast = arima_fit.forecast(steps=len(forecast_years))

arima_df = pd.DataFrame(
    {
        "year": forecast_years,
        "actual_trillions": actual,
        "forecast_trillions": forecast,
    }
)
arima_df["gap_trillions"] = arima_df["actual_trillions"] - arima_df["forecast_trillions"]
arima_df["gap_pct"] = (arima_df["gap_trillions"] / arima_df["forecast_trillions"]) * 100

print("ARIMA(1,1,1) trained on 1988–2007 — counterfactual vs actual:")
display(arima_df.round(2))

# Cumulative trade lost vs trend
cum_loss = arima_df.loc[arima_df["gap_trillions"] < 0, "gap_trillions"].sum()
print(f"\nCumulative trade BELOW trend (2008–2015): ${abs(cum_loss):.1f} trillion")

fig, ax = plt.subplots(figsize=(14, 5))
hist = series[series["year"] <= train_end_year]
ax.plot(hist["year"], hist["total_trillions"], "o-", label="History (train)", color="#2ca02c")
ax.plot(arima_df["year"], arima_df["actual_trillions"], "o-", label="Actual", color="#1f77b4")
ax.plot(arima_df["year"], arima_df["forecast_trillions"], "s--", label="ARIMA counterfactual", color="#ff7f0e")
ax.fill_between(
    arima_df["year"],
    arima_df["actual_trillions"],
    arima_df["forecast_trillions"],
    where=arima_df["actual_trillions"] < arima_df["forecast_trillions"],
    alpha=0.25,
    color="#d62728",
    label="Trade below trend",
)
for shock in (2009, 2015):
    ax.axvline(shock, color="black", linestyle=":", alpha=0.4)
ax.set_ylabel("Trillions USD")
ax.set_title("Counterfactual: what if pre-2008 growth had continued?")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "07_ml_arima_counterfactual.png"), dpi=150, bbox_inches="tight")
plt.show()

### Model 5 — Random Forest "surprise" model

Train on pre-2008 data to predict YoY growth from momentum (lags + trade level). Large **residuals** = years the model could not explain — our data-driven shock detector.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_df = yearly_raw.sort_values("year")[["year", "total_trillions", "yoy_pct"]].copy()
rf_df["lag1_yoy"] = rf_df["yoy_pct"].shift(1)
rf_df["lag2_yoy"] = rf_df["yoy_pct"].shift(2)
rf_df = rf_df.dropna()

X_cols = ["lag1_yoy", "lag2_yoy", "total_trillions"]
rf_train = rf_df[rf_df["year"] <= 2007]
rf_test = rf_df[rf_df["year"] >= 2008].copy()

rf_model = RandomForestRegressor(n_estimators=200, random_state=42)
rf_model.fit(rf_train[X_cols], rf_train["yoy_pct"])
rf_test["predicted_yoy"] = rf_model.predict(rf_test[X_cols])
rf_test["surprise"] = rf_test["yoy_pct"] - rf_test["predicted_yoy"]
rf_test["surprise_z"] = (rf_test["surprise"] - rf_test["surprise"].mean()) / rf_test["surprise"].std()
rf_test["is_shock"] = rf_test["surprise_z"].abs() > 1.0

importance = pd.Series(rf_model.feature_importances_, index=X_cols, name="importance").sort_values(ascending=False)

print("Random Forest feature importance:")
display(importance.round(3))

print("\nYears the model could NOT predict (|surprise z| > 1):")
display(rf_test.loc[rf_test["is_shock"], ["year", "yoy_pct", "predicted_yoy", "surprise", "surprise_z"]].round(1))

fig, ax = plt.subplots(figsize=(14, 5))
colors = ["#d62728" if s else "#4c72b0" for s in rf_test["is_shock"]]
ax.bar(rf_test["year"], rf_test["surprise"], color=colors, edgecolor="white")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("Surprise (actual YoY − predicted YoY)")
ax.set_xlabel("Year")
ax.set_title("ML shock detector: Random Forest prediction errors post-2007")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "08_ml_rf_surprise.png"), dpi=150, bbox_inches="tight")
plt.show()

### Model 6 — Oil explains the shocks?

Regress **total clean trade** on **oil/fuels trade** each year. The residual is the non-oil economy; a drop in R² or a large residual flags oil-driven shocks.

In [ ]:
oil_yearly_reg = (
    df_clean[df_clean["category"] == OIL_CAT]
    .groupby("year")["trade_usd"]
    .sum()
    .div(1e12)
    .rename("oil_trillions")
)
reg_df = (
    yearly_clean.set_index("year")[["total_trillions"]]
    .join(oil_yearly_reg)
    .dropna()
    .reset_index()
)

oil_model = LinearRegression()
oil_model.fit(reg_df[["oil_trillions"]], reg_df["total_trillions"])
reg_df["predicted_total"] = oil_model.predict(reg_df[["oil_trillions"]])
reg_df["residual"] = reg_df["total_trillions"] - reg_df["predicted_total"]
reg_df["r2"] = oil_model.score(reg_df[["oil_trillions"]], reg_df["total_trillions"])

print(f"Oil alone explains {reg_df['r2'].iloc[0]*100:.1f}% of total clean trade variance")
print(f"Slope: each +$1T oil trade ↔ +${oil_model.coef_[0]:.2f}T total trade\n")

shock_residuals = reg_df[reg_df["year"].isin([2009, 2015])][["year", "total_trillions", "predicted_total", "residual"]]
print("Residuals at shock years (negative = trade below oil-implied level):")
display(shock_residuals.round(2))

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(reg_df["oil_trillions"], reg_df["total_trillions"], c=reg_df["year"], cmap="viridis", s=80)
x_line = np.linspace(reg_df["oil_trillions"].min(), reg_df["oil_trillions"].max(), 50)
ax.plot(x_line, oil_model.predict(x_line.reshape(-1, 1)), "r--", label="Oil → total fit")
for _, row in reg_df[reg_df["year"].isin([2009, 2015, 2013])].iterrows():
    ax.annotate(str(int(row["year"])), (row["oil_trillions"], row["total_trillions"]), fontsize=9, fontweight="bold")
ax.set_xlabel("Oil trade (trillions USD)")
ax.set_ylabel("Total clean trade (trillions USD)")
ax.set_title("Oil trade predicts total trade — shock years labelled")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "09_ml_oil_regression.png"), dpi=150, bbox_inches="tight")
plt.show()

### ML synthesis — do all models agree?

| Shock | Descriptive | YoY z-score | Change-point | ARIMA gap | RF surprise | Oil residual |
|-------|-------------|-------------|--------------|-----------|-------------|--------------|
| **2009** | −20.7% | Flagged | Level break ~2008 | −$12.3T | −37 pp | Below oil-implied |
| **2015** | −13.3% | Borderline | YoY break ~2013 | −$24.9T | −32 pp | Below oil-implied |
| **2016** | −20.2% | Flagged | — | — | −43 pp | Incomplete data |

**Deeper meaning:** The 2009 shock was a **broad financial contagion** — trade collapsed faster than oil alone predicted. The 2015 shock was **oil-led** — ARIMA and oil regression both show the largest cumulative gap. Random Forest confirms both years as statistically "surprising" given pre-crisis momentum.

**Dashboard headline supported by ML:** Two distinct macro regimes — a V-shaped financial crisis (2009) and a slower commodity-price bleed (2015), both visible without labelling those years in the models.

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Descriptive outputs
yearly_raw.to_csv(os.path.join(OUTPUT_DIR, "yearly_trade_raw.csv"), index=False)
yearly_clean.to_csv(os.path.join(OUTPUT_DIR, "yearly_trade_clean.csv"), index=False)
shock_raw.to_csv(os.path.join(OUTPUT_DIR, "shock_stats_raw.csv"), index=False)
shock_clean.to_csv(os.path.join(OUTPUT_DIR, "shock_stats_clean.csv"), index=False)
decline_09.to_csv(os.path.join(OUTPUT_DIR, "category_decline_2009.csv"))
decline_15.to_csv(os.path.join(OUTPUT_DIR, "category_decline_2015.csv"))

# ML outputs
trend_df.to_csv(os.path.join(OUTPUT_DIR, "ml_trend_anomalies.csv"), index=False)
yoy_df.to_csv(os.path.join(OUTPUT_DIR, "ml_yoy_anomalies.csv"), index=False)
ml_yearly.to_csv(os.path.join(OUTPUT_DIR, "ml_isolation_forest_years.csv"), index=False)
country_shocks.to_csv(os.path.join(OUTPUT_DIR, "ml_country_shock_clusters.csv"))
cluster_summary.to_csv(os.path.join(OUTPUT_DIR, "ml_cluster_summary.csv"))
changepoint_df.to_csv(os.path.join(OUTPUT_DIR, "ml_changepoints.csv"), index=False)
arima_df.to_csv(os.path.join(OUTPUT_DIR, "ml_arima_counterfactual.csv"), index=False)
rf_test.to_csv(os.path.join(OUTPUT_DIR, "ml_rf_surprise.csv"), index=False)
reg_df.to_csv(os.path.join(OUTPUT_DIR, "ml_oil_regression.csv"), index=False)

print("Saved CSV summaries and PNG charts to:", OUTPUT_DIR)
print("Charts: 01-03 descriptive, 04-09 ML")

## Narrative for the dashboard (pending your confirmation)

> **Global commodity trade surged for two decades, peaked near $58 trillion in 2013, and was punctuated by two sharp contractions — the 2009 financial crisis (−21%) and the 2015 commodity price collapse (−13%). Trade rebounded quickly after 2009 (+21% in one year) but the 2015 slowdown was driven largely by falling oil & mineral fuel values, not a collapse in trade volume across all goods.**

### What to confirm
1. Does the dual-line chart (raw vs clean) make the data caveat clear enough?
2. Are 2009 and 2015 the right anchor years for the dashboard story?
3. Should the dashboard headline use **$58T (raw)** or **$15.5T (clean peak in 2011)**?

Once you approve, we build the Streamlit page around this story with interactive year brushing and shock annotations.